# Basic LocoClient locomotion

This notebook sends a short time-based path with `LocoClient`: walk forward for about 1 m, rotate 90 degrees, walk 1 m, rotate 90 degrees, and walk 1 m.

Time-based locomotion is approximate. Use a clear open area, keep a hand near the emergency stop, and tune speed values before running near people or obstacles.


In [1]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Configured for iface='eth0', domain_id=0.


Import the Unitree locomotion client and UI helpers.


In [2]:
import math
import threading
import time

import ipywidgets as widgets
from IPython.display import display

from unitree_sdk2py.core.channel import ChannelFactoryInitialize
from unitree_sdk2py.g1.loco.g1_loco_client import LocoClient


Create a small wrapper that starts/stops continuous velocity commands and always sends a stop packet at the end of a segment.


In [3]:
class LocoTimedPath:
    def __init__(self, iface="eth0", domain_id=0, timeout=10.0):
        ChannelFactoryInitialize(int(domain_id), str(iface))
        self.client = LocoClient()
        self.client.SetTimeout(float(timeout))
        self.client.Init()
        self._stop = threading.Event()
        self._thread = None

    def move(self, vx=0.0, vy=0.0, vyaw=0.0, duration_s=1.0, rate_hz=20.0):
        deadline = time.monotonic() + max(0.0, float(duration_s))
        dt = 1.0 / max(1.0, float(rate_hz))
        while time.monotonic() < deadline and not self._stop.is_set():
            self.client.Move(float(vx), float(vy), float(vyaw), continous_move=True)
            time.sleep(dt)
        self.stop()

    def stop(self):
        try:
            if hasattr(self.client, "StopMove"):
                self.client.StopMove()
            else:
                self.client.Move(0.0, 0.0, 0.0, continous_move=False)
        except Exception:
            pass

    def run_three_segment_path(self, distance_m=1.0, forward_speed=0.25, yaw_speed=0.45, log=print):
        self._stop.clear()
        forward_time = abs(float(distance_m)) / max(0.05, abs(float(forward_speed)))
        turn_time = (math.pi / 2.0) / max(0.05, abs(float(yaw_speed)))
        steps = [
            ("walk 1", forward_speed, 0.0, 0.0, forward_time),
            ("rotate 90", 0.0, 0.0, yaw_speed, turn_time),
            ("walk 2", forward_speed, 0.0, 0.0, forward_time),
            ("rotate 90", 0.0, 0.0, yaw_speed, turn_time),
            ("walk 3", forward_speed, 0.0, 0.0, forward_time),
        ]
        for label, vx, vy, vyaw, duration in steps:
            if self._stop.is_set():
                break
            log(f"{time.strftime('%H:%M:%S')} {label}: vx={vx:.2f} vyaw={vyaw:.2f} duration={duration:.2f}s")
            self.move(vx=vx, vy=vy, vyaw=vyaw, duration_s=duration)
            time.sleep(0.25)
        self.stop()
        log(f"{time.strftime('%H:%M:%S')} path complete or stopped")

    def start_background_path(self, **kwargs):
        if self._thread is not None and self._thread.is_alive():
            return "A path is already running."
        self._thread = threading.Thread(target=self.run_three_segment_path, kwargs=kwargs, daemon=True)
        self._thread.start()
        return "Path started."

    def request_stop(self):
        self._stop.set()
        self.stop()
        return "Stop requested."

loco_path = LocoTimedPath(IFACE, DOMAIN_ID)
print("LocoClient ready.")


LocoClient ready.


Run the panel. Start with low speeds and check the actual distance on your floor before increasing them.


In [4]:
distance = widgets.FloatSlider(value=1.0, min=0.2, max=2.0, step=0.05, description="Distance m")
forward_speed = widgets.FloatSlider(value=0.25, min=0.05, max=0.6, step=0.05, description="Forward")
yaw_speed = widgets.FloatSlider(value=0.45, min=0.1, max=1.0, step=0.05, description="Yaw rad/s")
start = widgets.Button(description="Run Path", button_style="success")
stop = widgets.Button(description="Stop", button_style="danger")
log_box = widgets.Textarea(value="", layout=widgets.Layout(width="100%", height="180px"), disabled=True)


def add_log(line):
    log_box.value = (line + "\n" + log_box.value)[:4000]


def on_start(_):
    msg = loco_path.start_background_path(
        distance_m=distance.value,
        forward_speed=forward_speed.value,
        yaw_speed=yaw_speed.value,
        log=add_log,
    )
    add_log(msg)


def on_stop(_):
    add_log(loco_path.request_stop())

start.on_click(on_start)
stop.on_click(on_stop)
display(widgets.VBox([widgets.HBox([distance, forward_speed, yaw_speed]), widgets.HBox([start, stop]), log_box]))
